# Lecture 4h: Random-Walk Metropolis-Hastings for a Correlation Posterior
**Simulations in Statistics (52001)**

---

## Overview

`Part4h(n, S11, S22, S12, a, burn.in, N, x0, seed)` simulates from the posterior distribution of the correlation parameter $\rho$ in a bivariate Normal model with Jeffreys prior.

The log-posterior kernel is

$$\log \pi(\rho) = -\frac{n+3}{2}\log(1-\rho^2)
- \frac{S_{11}+S_{22}-2\rho S_{12}}{2(1-\rho^2)}, \qquad -1 < \rho < 1.$$

The random-walk proposal is

$$Y_i = X_{i-1} + \epsilon_i, \qquad \epsilon_i \sim U(-a,a).$$

Because the proposal is symmetric, the Metropolis-Hastings acceptance probability is based only on the posterior ratio.

**Construction:**

1. Start from `x0 = 0`.
2. Run `burn.in` random-walk Metropolis-Hastings iterations without recording values.
3. Run `N` additional iterations and store the path in `X`.
4. Compute `mean(diff(X) != 0)` and `mean(X)`.


## R Implementation


In [1]:
log_posterior_rho <- function(rho, n, S11, S22, S12) {
  out <- rep(-Inf, length(rho))
  inside <- abs(rho) < 1
  rho.inside <- rho[inside]
  out[inside] <- -((n + 3) / 2) * log(1 - rho.inside^2) -
    (S11 + S22 - 2 * rho.inside * S12) / (2 * (1 - rho.inside^2))
  return(out)
}

Part4h <- function(n, S11, S22, S12, a, burn.in, N, x0 = 0, seed = 1) {
  if (!is.null(seed)) set.seed(seed)

  current <- x0
  log.current <- log_posterior_rho(current, n, S11, S22, S12)

  for (i in seq_len(burn.in)) {
    proposal <- current + runif(1, min = -a, max = a)
    log.proposal <- log_posterior_rho(proposal, n, S11, S22, S12)

    if (log(runif(1)) < log.proposal - log.current) {
      current <- proposal
      log.current <- log.proposal
    }
  }

  X <- numeric(N)
  for (i in seq_len(N)) {
    proposal <- current + runif(1, min = -a, max = a)
    log.proposal <- log_posterior_rho(proposal, n, S11, S22, S12)

    if (log(runif(1)) < log.proposal - log.current) {
      current <- proposal
      log.current <- log.proposal
    }

    X[i] <- current
  }

  return(X)
}


In [2]:
n <- 1000
S11 <- 923.7
S22 <- 931.04
S12 <- -505.42
a <- 0.098
burn.in <- 1000
N <- 1e6
x0 <- 0
seed <- 1

cat("n       =", n, "
")
cat("S11     =", S11, "
")
cat("S22     =", S22, "
")
cat("S12     =", S12, "
")
cat("a       =", a, "
")
cat("burn.in =", burn.in, "
")
cat("N       =", N, "
")
cat("x0      =", x0, "
")
cat("seed    =", seed, "
")


n       = 1000 
S11     = 923.7 
S22     = 931.04 
S12     = -505.42 
a       = 0.098 
burn.in = 1000 
N       = 1e+06 
x0      = 0 
seed    = 1 


## Computing the Results


In [3]:
X <- Part4h(
  n = n,
  S11 = S11,
  S22 = S22,
  S12 = S12,
  a = a,
  burn.in = burn.in,
  N = N,
  x0 = x0,
  seed = seed
)

mean_changed <- mean(diff(X) != 0)
mean_rho <- mean(X)

cat("a. mean(diff(X) != 0) =", mean_changed, "
")
cat("b. mean(X)            =", mean_rho, "
")


a. mean(diff(X) != 0) = 0.3117083 
b. mean(X)            = -0.5671035 
